In [4]:
import os
import sys
import shutil
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import pandas as pd
import numpy as np
source_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(source_path)

import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
import random
from torch.utils.data import DataLoader
import winsound
import itertools
import random
from datetime import datetime


In [5]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# grid init

In [34]:
set_seed(17) #i set the seeds for the grid search initializaton

custom_pretrained='original' #original
kind = 'patches_224'  # Example kind, can be changed
selected_FE = 'resnet50' #'clip-vit-large-patch14-un' 'BEiT-Large' 'BEiT-Large-inter'	
#'clip-vit-large-patch14-inter'# #'trocr-base-stage1'#'clip-vit-large-patch14'#'DeiT-Tiny' 
# #'clip-vit-large-patch14'  # Example feature extractor, can be changed
lang= 'ar' #'ar' #if en only train on en, if ar only train on ar, if en+ar or '' train on both

lang_sub = '_'+lang if lang in ['en', 'ar'] else ''
if custom_pretrained=='original':
    save_common=source_path+f'\\outputs\\online_deep_feature_extraction\\{selected_FE}\\representation_extraction\\torch_model_trained_on_rep{lang_sub}\\'
else:
    save_common=source_path+f'\\outputs\\online_deep_feature_extraction\\{selected_FE}\\{custom_pretrained}\\torch_model_trained_on_rep{lang_sub}\\'

clear_directory = True  # Set to True to clear the directory before saving checkpoints

extra_view=False
extra_integration_mode = 'concat'  # 'concat' or 'add'
data_augmentation = False

search_type = 'single_experiment' #'grid_search'  # or 'random_search'
model_list = ['MLPClassifier1','MLPClassifier2','MLPClassifier3']
best_result = [np.inf, np.inf,np.inf]
all_results = []
grid_search_params = {
    'lr': [1e-5,1e-4,1e-3],
    'dropout': [0.1, 0.4, 0.9],
    'n_neurons': [16,32,64,128,256,512],
    'model_name': ['MLPClassifier1'],
    'optimizer': ['Adam'],
    'scheduler': ['no_scheduling'],
    'log_grad_norm': [True],
    'activation': ['relu'],
    'with_input_norm': ['batch_norm']#,None],  # Whether to use input normalization
}
random_search_params = {    
    'lr': [1e-6,1e-5, 1e-4, 1e-3, 1e-2, 1e-1],
    'dropout': [0.1, 0.2, 0.4, 0.7,0.9],
    'n_neurons': [16, 32 , 64, 128, 256, 512],
    'model_name': ['MLPClassifier1', 'MLPClassifier2'],
    'optimizer': ['Adam','AdamW','SGD'],
    'scheduler': ['no_scheduling','OneCycleLR','CosineAnnealingLR','CyclicalLR', 'ReduceLROnPlateau','StepLR','CosineAnnealingWarmRestarts'],
    'log_grad_norm': [True, False],
    'activation': ['relu', 'tanh','leaky_relu'],
    'with_input_norm': [True,False],
}
#{'weight_decay': weight_decay}
#'no_scheduling'
#'CosineAnnealingLR' {'T_max': total_epochs, 'eta_min': lr_final}
#'OneCycleLR' {'total_epochs': total_epochs, 'steps_per_epoch': 705, 'max_lr':0.1}
# 'CosineAnnealingWarmRestarts'
single_experiment = {
    'lr': [1e-3],
    'dropout': [0.1],
    'n_neurons': [128], #not used if hiddden_sizes is used
    'hidden_sizes': [[16]],
    'model_name': ['MLPClassifier1'],
    'optimizer': ['Adam'],
    'scheduler': ['no_scheduling'],
    'log_grad_norm': [True],
    'activation': ['relu'],
    'with_input_norm': ['batch_norm'],#['batch_norm'],
}
if search_type == 'grid_search':
    param_grid = grid_search_params
elif search_type == 'random_search':
    param_grid = random_search_params
else:
    param_grid = single_experiment

keys, values = zip(*param_grid.items())
all_combos = list(itertools.product(*values))

# Shuffle combinations
random.shuffle(all_combos)
if search_type == 'random_search': 
    # Pick N random samples (e.g., 5)
    N = 100 if 100 < len(all_combos) else len(all_combos)
    experiments = all_combos[:N]
else:
    # For grid search, use all combinations
    experiments = all_combos[:]

In [35]:
file_id = '09-12'
prev_log = os.path.join(save_common, f'{search_type}_results_{file_id}.csv')
if os.path.exists(prev_log) and search_type != 'single_search':
    prev_results = pd.read_csv(prev_log)
    print(prev_results['best_val_loss'].min())
    print(prev_results.iloc[prev_results['best_val_loss'].idxmin()])
    #display(prev_results.iloc[prev_results['best_val_loss'].idxmin()])
    #print(prev_results['id'])
    for model in model_list:
        model_results = prev_results[prev_results['model_name'] == model]
        if not model_results.empty:
            best_result_temp = model_results['best_val_loss'].min()
            print(f"Best result for {model}: {best_result_temp}")
            best_result[model_list.index(model)]= best_result_temp
        else:
            print(f"No results found for model: {model}")
    last_index = prev_results['id'].max() if not prev_results.empty else 0
else:
    prev_results = None
    last_index = None
print(best_result)
print(f"Last index: {last_index}")
print(search_type)
print(len(experiments))
print(experiments)

[inf, inf, inf]
Last index: None
single_experiment
1
[(0.001, 0.1, 128, [16], 'MLPClassifier1', 'Adam', 'no_scheduling', True, 'relu', 'batch_norm')]


# run

In [36]:
suffix = '_augmented' if data_augmentation else ''
train_filename,val_filename, _ = file_IO.load_input_files(source_path,selected_FE,kind,suffix, custom_pretrained=custom_pretrained)
if extra_view:
    extra_train_filename, extra_val_filename, _ = file_IO.load_input_files(source_path,selected_FE,kind='body',suffix='', custom_pretrained=custom_pretrained)
else:
    extra_train_filename, extra_val_filename = None, None

loss_criterion = 'CrossEntropyLoss'
total_epochs = 100
use_profiler = False
profiler_config = None
plot_every = 1
patience = 2
run_epochs = total_epochs
use_amp = False
val_percentage = 1.0
batch_size = 64
aggregation_mode = None  # 'mean' or 'max'
weight_decay = 1e-2  # Weight decay for the optimizer 
lr_final = 1e-8  # Final learning rate for the backbone

# Assign unique IDs
start = last_index + 1 if last_index is not None else 0
for i, combo in enumerate(experiments[start:]):
    #i set the seeds for the grid search initializaton
    print('Setting seed')
    set_seed(42) 
    experiment_dict = dict(zip(keys, combo))
    #skip some combinations
    '''if experiment_dict['model_name']== 'MLPClassifier2' and experiment_dict['n_neurons'] >= 128:
        continue
    if experiment_dict['model_name']== 'MLPClassifier1' and  experiment_dict['with_input_norm'] == None:
        continue'''
    experiment_dict['id'] = i+start
    
    #define checkpoints directory for the current experiment, clear if needed
    model_name = experiment_dict['model_name']
    save_path = save_common+f'{model_name}'
    file_IO.access_or_create_dir(save_path)
    checkpoint_path=save_path+'\\checkpoints'
    file_IO.access_or_create_dir(checkpoint_path)
    if clear_directory==True:
        print(f'Clearing directory: {checkpoint_path}')
        file_IO.clear_folder(checkpoint_path)

    #get experiment parameters
    log_grad_norm = experiment_dict['log_grad_norm']
    lr = experiment_dict['lr']  # Learning rate for the optimizer
    with_input_norm = experiment_dict.get('with_input_norm', None)  # Whether to use input normalization
    #for full fine tuning
    optimizer_phases = [total_epochs]  
    optim_config = {
        'optimizer_phases':optimizer_phases,  
        'type_of_training': 'from_scratch',
        'scheduling': experiment_dict['scheduler'],  
        'optimizer_name':experiment_dict['optimizer'], 
        'phase_lr': [lr],
        'phase_optimizer_hyperparams': [{'weight_decay': weight_decay}],
        'phase_scheduler_hyperparams': [{'T_max': total_epochs, 'eta_min': lr_final, 'total_epochs': total_epochs,
                                          'steps_per_epoch': 705, 'max_lr':0.01,'step_size': 10,'patience':int(patience/2),
                                          'max_lr_cycle':0.001, 'base_lr_cycle': 0.0001}],
    }
    if optim_config['scheduling'][0] in ['OneCycleLR','CyclicalLR']:
        step_at_epoch = True
    else:
        step_at_epoch = False
    
    #set all experiment parameters
    args = script_launching.DotDict(
        data_augmentation=data_augmentation,
        extra_view=extra_view,
        loss_criterion=loss_criterion,
        model_name=model_name,
        total_epochs=total_epochs,
        patience=patience,
        log_grad_norm=log_grad_norm,
        use_amp = use_amp,
        batch_size=batch_size,
        val_percentage=val_percentage,
        weight_decay=weight_decay,
        lr=lr,
        lr_final=lr_final,
        optim_config=optim_config,
        aggregation_mode=aggregation_mode,
        extra_integration_mode=extra_integration_mode,
        train_filename=train_filename,
        val_filename=val_filename,
        extra_train_filename=extra_train_filename,
        extra_val_filename=extra_val_filename,
        step_at_epoch=step_at_epoch,
        experiment_id=experiment_dict['id'],
        with_input_norm=with_input_norm,
    )
    # Save the arguments to a file in the checkpoint path
    file_IO.save_args(args,checkpoint_path)  
    
    # Define datasets and group by page
    train_df = pd.read_csv(train_filename)
    val_df = pd.read_csv(val_filename)

    if lang == 'en':
        train_df = train_df[train_df['isEng'] == 1]
        val_df = val_df[val_df['isEng'] == 1]
    elif lang == 'ar':
        train_df = train_df[train_df['isEng'] == 0]
        val_df = val_df[val_df['isEng'] == 0]
    
    '''#print(len(train_df.columns))
    #cols_to_drop = [c for c in val_df.columns if c.startswith('f') and len(c) > 1 and c[1].isdigit()][256:]
    #train_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
    #val_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
    #print(len(train_df.columns))'''
    #get normalization parameters if you use the training set to normalize the inputs
    mean,scale = model_utils.get_normalization_parameters(train_filename)

    if extra_view:
        train_df_extra = pd.read_csv(extra_train_filename)
        val_df_extra = pd.read_csv(extra_val_filename)
        train_df = dataframes.merge_dfs(train_df, train_df_extra, mode=extra_integration_mode)
        val_df = dataframes.merge_dfs(val_df, val_df_extra, mode=extra_integration_mode)

    train_df = dataframes.aggregate_dfs(train_df,mode=aggregation_mode)
    val_df = dataframes.aggregate_dfs(val_df,mode=aggregation_mode)
    #train_df=file_IO.change_filename_from_to(train_df, fr=saved, to=running
    #cols_to_drop = [c for c in train_df.columns if not(c.startswith('f') and len(c) > 1 and c[1].isdigit())]
    cols_to_keep = [c for c in train_df.columns if c.startswith('f') and len(c) > 1 and c[1].isdigit()]
    in_features = len(cols_to_keep)  # Number of features from the model output

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device is: ",device)

    train_dataset = dataframes.CustomExtractedDataset(train_df, label_column='male')
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dataset = dataframes.CustomExtractedDataset(val_df, label_column='male')
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    #if i spcify the neurons for each hidden layer i pass kwargs to the get_classification_head function
    if 'hidden_sizes' in experiment_dict:
        kwargs = {'hidden_sizes': experiment_dict['hidden_sizes']}
    else:
        kwargs = {}
    loss_fn = training_utils.get_criterion(name=loss_criterion)
    model = model_utils.get_classification_head(name=model_name, in_features=in_features, num_classes=2,
                                                dropout=experiment_dict['dropout'], n_neurons=experiment_dict['n_neurons'],
                                                activation=experiment_dict['activation'],with_input_norm=with_input_norm, 
                                                mean=mean,scale=scale,**kwargs)
    #print model for debugging
    print(model)

    best_model_performance=training_utils.train_fine(
        model=model,
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        device=device,
        total_epochs=total_epochs,
        loss_fn=loss_fn,
        use_profiler=use_profiler,
        profiler_config=profiler_config,
        save_path=save_path,
        plot_every=plot_every,
        early_stopping_patience=patience,
        checkpoint_path=checkpoint_path+"\\checkpoint.pt",
        log_grad_norm=log_grad_norm,
        run_epochs=run_epochs,
        use_amp=use_amp,
        val_percentage=val_percentage,  # Use 10% of validation data for linear evaluation
        optim_config=optim_config,  # e.g., 'Adam', 'SGD', 'AdamW'
        save_backbone=False,
        step_at_epoch=step_at_epoch,
        # ... other parameters
    )

    #i load the best model and compute the page-level metrics
    checkpoint_temp = os.path.join(checkpoint_path, 'checkpoint_best.pt')
    checkpoint = torch.load(checkpoint_temp, map_location=torch.device('cpu'), weights_only=False)
    model = model.to('cpu')
    model.load_state_dict(checkpoint['model_state_dict'])
    val_df['train']=0
    res_df,out_results=evaluation_utils.compute_predictions_and_uncertainties(model, val_df, head_type='pytorch',calibrate=False,
                                                                              threshold=0.5, list_of_metrics=['majority_vote', 'weighted_vote', 'most_probable'])
    
    #i add the results stored in the output of the training and evaluation function to the experiment properties
    for key in experiment_dict.keys():
        if key not in best_model_performance:
            best_model_performance[key] = experiment_dict[key]
    for key in out_results.keys():
        if key not in best_model_performance:
            best_model_performance[key] = out_results[key]
    all_results.append(best_model_performance)

    #always run in single_search
    index=model_list.index(model_name)
    if best_model_performance['best_val_loss'] < best_result[index]:
        best_checkpoint = os.path.join(checkpoint_path, "checkpoint_best.pt")
        destination = os.path.join(save_path, "checkpoint_best.pt")
        shutil.copy2(best_checkpoint, destination)
        best = os.path.join(checkpoint_path, "training_plot.png")
        destination = os.path.join(save_path, "training_plot.png")
        shutil.copy2(best, destination)
        best = os.path.join(checkpoint_path, "args.txt")
        destination = os.path.join(save_path, "args.txt")
        shutil.copy2(best, destination)
        best_result[index] = best_model_performance['best_val_loss']

    all_results_df = pd.DataFrame(all_results)
    if prev_results is not None:
        all_results_df = pd.concat([prev_results, all_results_df], ignore_index=True)
    current_date = datetime.now().strftime('%d-%m')
    all_results_df.to_csv(os.path.join(save_common, f'{search_type}_results_{current_date}.csv'), index=False)
    '''duration = 3000  # milliseconds
    freq = 880  # Hz
    winsound.Beep(freq, duration)'''

Setting seed
Clearing directory: c:\Users\andre\VsCode\PD related projects\gender_detection\outputs\online_deep_feature_extraction\resnet50\representation_extraction\torch_model_trained_on_rep_ar\MLPClassifier1\checkpoints
aggregating patches, length before: 22560
length after: 22560
aggregating patches, length before: 5680
length after: 5680
Device is:  cuda
Extracted 2048 feature columns:
Extracted 2048 feature columns:
CustomMLP(
  (model): Sequential(
    (0): BatchNorm1d(2048, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (1): Linear(in_features=2048, out_features=16, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.1, inplace=False)
    (4): Linear(in_features=16, out_features=2, bias=True)
  )
)
2026-05-22 11:38:53,966 - INFO - Model size: 0.17 MB
📂 No checkpoint found at c:\Users\andre\VsCode\PD related projects\gender_detection\outputs\online_deep_feature_extraction\resnet50\representation_extraction\torch_model_trained_on_rep_ar\MLPClassifier1\checkpoints

Epoch 1/100 [Val]: 100%|██████████| 89/89 [00:01<00:00, 52.68it/s]

2026-05-22 11:39:02,177 - INFO - Epoch 1| Train Accuracy 0.6807| Train Loss: 0.5914 | Val Acc: 0.6419 | Val Loss: 0.6218 | Avg Grad Norm: 1.0360 | Epoch Time: 8.16s | Val Time: 1.71s
2026-05-22 11:39:02,182 - INFO - block 0 lr: 0.001000
2026-05-22 11:39:02,202 - INFO - ✅ Saved new best model at epoch 1
2026-05-22 11:39:02,219 - INFO - ⏳ No improvement for 0 epoch(s)


2026-05-22 11:39:04,961 - INFO - 
Epoch 1 - Optimization Phase: 0


Epoch 2/100 [Val]: 100%|██████████| 89/89 [00:01<00:00, 55.41it/s]

...

2026-05-22 11:39:40,799 - INFO - Epoch 4| Train Accuracy 0.7102| Train Loss: 0.5629 | Val Acc: 0.6467 | Val Loss: 0.6212 | Avg Grad Norm: 0.4824 | Epoch Time: 10.55s | Val Time: 1.67s
2026-05-22 11:39:40,799 - INFO - block 0 lr: 0.001000
2026-05-22 11:39:40,823 - INFO - ⏳ No improvement for 1 epoch(s)


2026-05-22 11:39:43,077 - INFO - 
Epoch 4 - Optimization Phase: 0


Epoch 5/100 [Val]: 100%|██████████| 89/89 [00:01<00:00, 57.46it/s]

2026-05-22 11:39:53,280 - INFO - Epoch 5| Train Accuracy 0.7031| Train Loss: 0.5742 | Val Acc: 0.6468 | Val Loss: 0.6198 | Avg Grad Norm: 0.3738 | Epoch Time: 10.20s | Val Time: 1.57s
2026-05-22 11:39:53,284 - INFO - block 0 lr: 0.001000
2026-05-22 11:39:53,303 - INFO - ⏳ No improvement for 2 epoch(s)
2026-05-22 11:39:53,311 - INFO - ⛔ Early stopping at epoch 5 (no improvement for 2 epochs)



📊 Performance Summary:
Average batch time: 0.0131s
Peak GPU memory usage: 17.95 MB
2


In [37]:
all_results

[{'best_val_loss': 0.6101422706120451,
  'best_val_acc': 0.6695422535211267,
  'best_epoch': 2,
  'best_train_loss': 0.556845934212039,
  'best_train_acc': 0.7125886524822695,
  'last_epoch': 4,
  'last_val_loss': 0.619804679675841,
  'last_val_acc': 0.6468309859154929,
  'last_train_loss': 0.5741713900572856,
  'last_train_acc': 0.7030585106382978,
  'lr': 0.001,
  'dropout': 0.1,
  'n_neurons': 128,
  'hidden_sizes': [16],
  'model_name': 'MLPClassifier1',
  'optimizer': 'Adam',
  'scheduler': 'no_scheduling',
  'log_grad_norm': True,
  'activation': 'relu',
  'with_input_norm': 'batch_norm',
  'id': 0,
  'Accuracy for individual patches': 0.6695422535211267,
  'Accuracy for majority_vote': 0.6901408450704225,
  'Accuracy for weighted_vote': 0.6971830985915493,
  'Accuracy for most_probable': 0.7112676056338029,
  'Accuracy for writer level prediction': 0.704225352112676}]

# reload

In [6]:
def reload_modules():
    import importlib
    import utils.data_loading as data_loading
    import utils.visualization as visualization
    import utils.dataframes as dataframes
    import utils.utils_transforms as u_transforms
    import utils.training_utils as training_utils
    import utils.model_utils as model_utils
    import utils.file_IO as file_IO
    import utils.vit_rollout_mod as vit_rollout_mod
    import utils.script_launching as script_launching
    import utils.evaluation_utils as evaluation_utils
    
    importlib.reload(file_IO)
    importlib.reload(data_loading)
    importlib.reload(visualization)
    importlib.reload(dataframes)
    importlib.reload(u_transforms)
    importlib.reload(model_utils)
    importlib.reload(training_utils)
    importlib.reload(vit_rollout_mod)
    importlib.reload(script_launching)
    importlib.reload(evaluation_utils)

    return data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO, vit_rollout_mod, script_launching, evaluation_utils
data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO, vit_rollout_mod, script_launching, evaluation_utils = reload_modules()